# Naloga

---
## Analitični del

Za sistem izberemo normirane parametre iz učbenika: $a = 1{,}0$, $b = 1{,}0$ ter dušenje $d = 0{,}3$.

1. Zapišite sistem dveh diferencialnih enačb prvega reda za $x$ in $v = \dot{x}$:

   $\dot{x} =$ ____________________  $\dot{v} =$ ______________________________

2. Določite numerične koordinate vseh treh stacionarnih točk $(x^{\ast}, v^{\ast})$ v fazni ravnini:

   $T_0^{\ast} = (0, 0)$, $T_1^{\ast} = ($ ______ $, 0)$, $T_2^{\ast} = ($ ______ $, 0)$.

3. Zapišite splošno Jacobijevo matriko sistema $\mathbf{J}(x, v)$ ter jo ovrednotite v točkah $T_0^{\ast}$ in $T_1^{\ast}$:

   $\mathbf{J}(0, 0) =$ __________  $\mathbf{J}(1, 0) =$ __________

4. Za točko $T_1^{\ast} = (1, 0)$ izračunajte sled $\tau$, determinanto $\Delta$ in lastni vrednosti $\lambda_{1,2}$. Določite tip ravnotežja:

   $\tau =$ ______, $\Delta =$ ______ $\implies$ Tip: ____________________

---
## Naloga 1: Profil potenciala in nivojnice energije

1. Izrišite profil potencialne energije $V(x) = -\frac{a}{2}x^2 + \frac{b}{4}x^4$ na območju $x \in [-2, 2]$. Označite oba minimuma in vrh pregrade pri $x=0$.
2. Za primer brez dušenja ($d = 0$) narišite nivojnice skupne mehanske energije $E(x, v) = \frac{1}{2}v^2 + V(x) = \text{konst.}$ v fazni ravnini s funkcijo `matplotlib.pyplot.contour`. Posebej poudarite separatriso z energijo $E = 0$.

## Naloga 2: Fazni portret dušenega sistema in konvergence

1. Vključite dušenje $d = 0{,}3$. Na območju $(x, v) \in [-2, 2] \times [-2, 2]$ izrišite vektorsko polje toka s funkcijo `matplotlib.pyplot.streamplot`.
2. Z numeričnim integratorjem (npr. `scipy.integrate.solve_ivp`) simulirajte časovni odziv za začetna pogoja iz učbenika:
   - Trajektorija A: $(x_0, v_0) = (-1{,}5,\ 0{,}5)$
   - Trajektorija B: $(x_0, v_0) = (1{,}5,\ -0{,}5)$

   Obe trajektoriji vrišite v fazno ravnino in potrdite, da konvergirata vsaka v svoj stabilni jašek.
3. Izrišite časovni potek mehanske energije $E(t)$ in preverite, da se energija monotono zmanjšuje proti $V_{\min} = -0{,}25$.

## Naloga 3: Kartiranje bazenov privlačnosti

Za mrežo začetnih pogojev $(x_0, v_0) \in [-2, 2] \times [-2, 2]$ z ločljivostjo vsaj $100 \times 100$ točk določite končno stanje po dovolj dolgem času integracije ($t_{\text{end}} \approx 30$).

- Če se trajektorija ustali v levem minimumu ($x^{\ast} \approx -1$), točko obarvajte modro.
- Če se trajektorija ustali v desnem minimumu ($x^{\ast} \approx +1$), točko obarvajte rdeče.
- Izrišite dobljeno karto in opazujte mejo med bazenoma, ki poteka skozi sedelno točko $(0,0)$.

| Začetni pogoj $(x_0, v_0)$ | Končna lega $x(\infty)$ | Končna energija $E(\infty)$ | Bazen privlačnosti |
|:-:|:-:|:-:|:-:|
| $(-1{,}5,\ 0{,}5)$ | | | Levi / Desni |
| $(1{,}5,\ -0{,}5)$ | | | Levi / Desni |
| $(0{,}1,\ 0{,}0)$ | | | Levi / Desni |
| $(-0{,}1,\ 0{,}0)$ | | | Levi / Desni |
| $(0{,}0,\ 1{,}2)$ | | | Levi / Desni |

# Imports

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import sympy as sp
from scipy.integrate import solve_ivp

# Global parameters
a, b = 1.0, 1.0
d = 0.3              # dusenje
meja = 2.0           # obmocje [-meja, meja]^2
t_end = 30.0
N_mreza = 200        # locljivost karte bazenov

y_A = [-1.5, 0.5]
y_B = [1.5, -0.5]
tabela_zp = [(-1.5, 0.5), (1.5, -0.5), (0.1, 0.0), (-0.1, 0.0), (0.0, 1.2)]

rtol, atol = 1e-10, 1e-12

# Funkcije

In [9]:
# Simbolno: potencial, sistem 1. reda, ravnotezja, Jacobijeva matrika


In [10]:
def rhs(t, y, d):
    x, v = y
    return [v, a * x - b * x**3 - d * v]

def trajektorija(y0, t_end, d=d, n=3000):
    t_eval = np.linspace(0, t_end, n)
    return solve_ivp(rhs, (0, t_end), y0, args=(d,), t_eval=t_eval, rtol=rtol, atol=atol)

def rk4_mreza(X, Vv, t_end, d=d, dt=0.01):
    # RK4 hkrati za vse tocke mreze (vektorsko)
    def f(X, Vv):
        return Vv, a * X - b * X**3 - d * Vv
    for _ in range(int(t_end / dt)):
        k1x, k1v = f(X, Vv)
        k2x, k2v = f(X + dt / 2 * k1x, Vv + dt / 2 * k1v)
        k3x, k3v = f(X + dt / 2 * k2x, Vv + dt / 2 * k2v)
        k4x, k4v = f(X + dt * k3x, Vv + dt * k3v)
        X = X + dt / 6 * (k1x + 2 * k2x + 2 * k3x + k4x)
        Vv = Vv + dt / 6 * (k1v + 2 * k2v + 2 * k3v + k4v)
    return X, Vv

def klasifikacija(A, eps=1e-12):
    tau, Delta = np.trace(A), np.linalg.det(A)
    D = tau**2 - 4 * Delta
    if Delta < -eps:
        return "sedlo"
    if abs(Delta) < eps:
        return "degenerirano (Delta = 0)"
    if abs(tau) < eps:
        return "center"
    if D > eps:
        return "stabilen vozel" if tau < 0 else "nestabilen vozel"
    if D < -eps:
        return "stabilno gorišče" if tau < 0 else "nestabilno gorišče"
    return "degeneriran vozel"

def stabilna_mnogoterost(d=d, t_max=60.0):
    # integracija nazaj v casu iz sedla vzdolz stabilne lastne smeri
    lam_s = (-d - np.sqrt(d**2 + 4 * a)) / 2
    v_s = np.array([1.0, lam_s]) / np.hypot(1.0, lam_s)

    def izven(t, y, d):
        return 2 * meja - np.max(np.abs(y))
    izven.terminal = True

    veji = []
    for s in (1, -1):
        sol = solve_ivp(rhs, (0, -t_max), s * 1e-6 * v_s, args=(d,), events=izven,
                        rtol=rtol, atol=atol, max_step=0.01)
        veji.append(sol.y)
    return veji

# Testno območje

In [11]:
# Pripravljalna naloga


# Naloga 1

# Naloga 2

# Naloga 3

# Razmislek 1

In [12]:
# Minimalna v0 v (-1, 0), da delec preskoci pregrado (doseze x = 0), bisekcija


# Razmislek 2

In [13]:
# Tocka na meji bazenov (stabilna mnogoterost) + majhna motnja v v


# Razmislek 3

In [14]:
# Linearni oscilator: ena sama ravnotezna tocka, paraboličen potencial
